In [ ]:
# Preliminary Product Validation Report - GMLP Product Quality (Murcia SAOCOM)
# Copyright 2026 B-Open
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


## Introduction and Scope
The objective of this notebook is to provide evidence for the validation of the **GMLP products** generated by the COP ARD Processors for SAOCOM data over Murcia.

In particular, this notebook presents the evaluation of the phase contained in the GMLP products by assessing the absence of a long-term phase drift in comparison with the phase evolution over persistent scatterers (PS) and the phase evolution provided by an alternative technique (EMI), as defined in the Software Verification Plan (Reference: `esa-rose-l-015-D15`, Issue 1.1).

The GMLP product obtained in the test area in Murcia is evaluated in three spatial extents:

-   **Whole scene**
-   **Zone with a known very strong subsidence** (around Lorca village)
-   **Stable zone** without deformation

## Input Products
The dataset includes the following SAOCOM stacks and products acquired in the Murcia region:

1. **CSLC**
   `s3://bopen-sar/ARD-ROSE-L/SAOCOM/Murcia/ASC/S4/OUTPUT/CSLC`
2. **CMLP**
   `s3://bopen-sar/ARD-ROSE-L/SAOCOM/Murcia/ASC/S4/OUTPUT/CMLP`
3. **PS Mask**
   `s3://bopen-sar/ARD-ROSE-L/SAOCOM/Murcia/ASC/S4/OUTPUT/PS_mask`

## TC-ST-PERF-009: GMLP Product Quality (Murcia SAOCOM)

### Test Specification & Verification Criteria

<table class="table table-striped table-bordered" style="width:100%;">
  <thead>
    <tr>
      <th style="width: 22%;">Field</th>
      <th style="width: 78%;">Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><strong>Test case IDs</strong></td>
      <td>TC-ST-PERF-009</td>
    </tr>
    <tr>
      <td><strong>Objective</strong></td>
      <td>
        To evaluate the quality of the phase contained in the GMLP products generated by COP ARD:
        <ul style="margin-bottom: 0; padding-left: 20px;">
          <li>Evaluate the phase of GMLP by comparing it against an alternative technique (EMI).</li>
          <li>Evaluates the phase of GMLP by comparing it against quality points, i.e. permanent scatterers (PS).</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>Test Data</strong></td>
      <td>GMLP products (temporal stacks) generated from SAOCOM stripmap stacks in the Murcia test site.</td>
    </tr>
    <tr>
      <td><strong>Pre-conditions</strong></td>
      <td>
        <ul style="margin-bottom: 0; padding-left: 20px;">
          <li>CSLC and CMLP products generated for all dates.</li>
          <li>Mask of PS candidates generated from the CSLC product.</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>Procedure (EMI)</strong></td>
      <td>
        <ul style="margin-bottom: 0; padding-left: 20px;">
          <li>Compute the EMI-derived interferograms from the CSLC product using the same multi-look as the CMLP.</li>
          <li>Compute the coherence and phase of the interferograms formed by subtracting the phases of CMLP and EMI for all interferogram pairs.</li>
          <li>Estimate the velocity (drift rate) for each pixel that maximises the coherence between CMLP and EMI products.</li>
          <li>Plot and evaluate the spatial distribution and statistics of the velocity drift estimated.</li>
          <li>Plot and evaluate the spatial distribution and statistics of the coherence and phase with and without using the velocity drift estimated.</li>
          <li>Compute and accumulate in form of time series the lag-1 differences in the interferograms provided by CMLP and EMI.</li>
          <li>Plot and evaluate the average time series at the whole scene and at specific zones (high subsidence and stable).</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>Procedure (PS)</strong></td>
      <td>
        <ul style="margin-bottom: 0; padding-left: 20px;">
          <li>Compute the interferograms from the CSLC product and average the values around the PS locations (window size equal to multi-look size for CMLP), provided that the proportion of PS in the window is greater than 10%.</li>
          <li>Compute the coherence and phase of the interferograms formed by subtracting the phases of CMLP and PS-averaged products for all interferogram pairs.</li>
          <li>Estimate the velocity (drift rate) for each pixel that maximises the coherence between CMLP and PS-averaged products.</li>
          <li>Plot and evaluate the spatial distribution and statistics of the velocity drift estimated.</li>
          <li>Plot and evaluate the spatial distribution and statistics of the coherence and phase with and without using the velocity drift estimated.</li>
          <li>Compute and accumulate in form of time series of the lag-1 differences in the interferograms provided by CMLP and PS-averaged products.</li>
          <li>Plot and evaluate the average time series at the whole scene.</li>
        </ul>
      </td>
    </tr>
    <tr>
      <td><strong>Pass Criteria</strong></td>
      <td>
        <ul style="margin-bottom: 0; padding-left: 20px;">
          <li>The spatial distribution of phase difference, coherence and velocity drift should not show any pattern.</li>
          <li>The mean velocity drift should be less than <strong>1 mm/year</strong>.</li>
        </ul>
      </td>
    </tr>
  </tbody>
</table>

In [2]:
### Implementation: GMLP-VAL-SAO-EMI-001 (GMLP vs EMI) GMLP-VAL-SAO-PS-001 (GMLP vs PS)
# Execute the procedure steps to evaluate GMLP against the EMI alternative technique and PS.

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from cop_ard_app.io import read, write, glob, Path
from cop_ard_app.computation import create_client
from cop_ard_core.utils import get_interf
from scipy.constants import speed_of_light

In [ ]:
compute_interf = False
compute_expand = False
compute_emi = False

interf_emi_path = "/home/juanma/MyStore/Processed_Data/SAOCOM/ARD/Murcia/OUTPUT/interf_emi"
interf_ps_path = "/home/juanma/MyStore/Processed_Data/SAOCOM/ARD/Murcia/OUTPUT/interf_ps"

output_plot_dir = "/home/juanma/MyStore/Processed_Data/SAOCOM/ARD/Murcia/OUTPUT/plots"

posting = {"slant_range_time": 5, "azimuth_time": 10}

In [ ]:
# Dask cluster activation

In [ ]:
import dask

dask.config.set({"temporary-directory": "/home/juanma/MyStore/Temp/scratch/external/dask-temp"})

client = create_client("LOCAL", 16, 0)
#client = create_client("LOCAL", 1, 0, threads_per_worker=4, processes=False)


In [ ]:
# Support functions

In [ ]:
def emi_core(C_matrix):
    N = C_matrix.shape[0]  # assune square matrix
    try:
        W = np.linalg.inv(np.abs(C_matrix))
        eigenvalues, eigenvectors = np.linalg.eigh(C_matrix * W)
        v = eigenvectors[:, 0]
    except:
        v = np.zeros([N])
    v = v / np.abs(v)
    v = v * np.conj(v[0])
    return np.conj(v)

In [ ]:
def emi_phase_estimation(window):
    try:
        n_acq, w_rg, w_az = window.shape
        data = np.zeros([n_acq, w_rg * w_az], dtype=np.complex64)
        ind = np.full(n_acq, True)
        interf_emi = np.full(n_acq, np.nan, dtype=np.complex64)  # default output

        # loading data and checking for non-valid acquisitions
        for n in range(0, n_acq):
            data[n, :] = window[n, :, :].flatten()
            # ind[n] = (~np.isnan(data[n,:])).all() # check that all data are valid
            ind[n] = (data[n, :] != 0).all()  # check that all data are valid

        N_ = sum(ind)  # number of valid acquisitions
        if N_ != 0:
            data = data[ind, :]  # eliminate non-valid acquisitions
            cov = np.matmul(data.conj(), data.T)
            interf_emi_ = emi_core(cov + 0.1 * np.linalg.eigvalsh(cov)[N_ - 1] * np.eye(N_))

            # filling expanded output
            interf_emi[ind] = interf_emi_
    except:
        interf_emi = np.full(window.shape[0], np.nan, dtype=np.complex64)
        
    return interf_emi

In [ ]:
def get_multilook(a: xr.DataArray, posting: dict[str, int]):
    """
    Compute the complex interferogram between two SLCs and multilook it with
    the given posting factors.
    """
    multilook = {k: v * 2 for k, v in posting.items()}
    win_dims = {k: f"win_{k}" for k in multilook}
    slices = {k: slice(v - 1, None, posting[k]) for k, v in multilook.items()}

    # 1. Compute the complex interferogram

    # 3. Build the rolling view, decimate, and calculate the mean efficiently
    a_view = a.rolling(multilook).construct(win_dims)
    a_view_sel = a_view.isel(slices)
    a_rolling = a_view_sel.mean(dim=list(win_dims.values()))

    return a_rolling

In [ ]:
def expand_cslc(cslc: xr.DataArray, posting: dict[str, int]):
    """
    Compute the complex interferogram between two SLCs and multilook it with
    the given posting factors.
    """
    multilook = {k: v * 2 for k, v in posting.items()}
    win_dims = {k: f"win_{k}" for k in multilook}
    slices = {k: slice(v - 1, None, posting[k]) for k, v in multilook.items()}

    # 1. Compute the complex interferogram

    # 3. Build the rolling view, decimate, and calculate the mean efficiently
    cslc_view = cslc.rolling(multilook).construct(win_dims)
    cslc_expanded = cslc_view.isel(slices)

    return cslc_expanded

In [ ]:
# return the velocity maximizing the temporal coherence
def get_cohe_vel(cslc_diff, dv, time_y, wavelength):
    # transform velocity into phase correction
    corr_rad = dv * time_y * (4 * np.pi / wavelength)
    cohe_max = xr.zeros_like(cslc_diff.isel(acquisition_date=0), dtype=float)
    vel_max = xr.zeros_like(cslc_diff.isel(acquisition_date=0), dtype=float)
    cslc_diff_norm = (cslc_diff / np.abs(cslc_diff)).compute().chunk({"acquisition_date": 1, "azimuth_time": 256, "slant_range_time": 256})
    # teast all velocities
    for k,v in enumerate(dv.values):
        # comute temporal coherence
        cohe = np.abs((cslc_diff_norm * xr.ufuncs.exp(-1j * corr_rad.isel(vel=k))).mean(dim="acquisition_date")).compute()

        # check coherence value
        mm = cohe > cohe_max

        # return the best velocity and coherence
        vel_max = xr.where(mm, v, vel_max).compute()
        cohe_max = xr.where(mm, cohe, cohe_max).compute()

    return cohe_max, vel_max

In [ ]:
# Read data

In [ ]:
# Input CMLP & CSLC product path
base_dir = Path("/home/juanma/MyStore/Processed_Data/SAOCOM/ARD/Murcia/")

cslc_dir_name = base_dir.append("OUTPUT/CSLC")
cmlp_dir_name = base_dir.append("OUTPUT/CMLP")
ps_dir_name = base_dir.append("OUTPUT/PS_mask")



In [ ]:
# Retrieve wavelength: this should be done differently, as we read a product different from the CMLP or GMLP.

annot_path = Path("/home/juanma/my_s3_mount/SAOCOM/Murcia/ASC/S4/REF_DB/reference_db_static/data/EOSAO1B_2025-12-02T05:54:05.643823/reference/measurements/")

annot = read(annot_path, consolidated=False).slc.attrs['VV']
f0 = annot["RMA"]["INCA"]["FreqZero"]
wavelength = speed_of_light / f0 * 1000. # this is  lambda = c / f0  (in mm)

In [ ]:
# finding the directories containing the products
cslc_dirs = sorted([p.path for p in cslc_dir_name.glob("*.zarr/*CSL*/measurements")])
cmlp_dirs = sorted([p.path for p in cmlp_dir_name.glob("*.zarr/cmlp*/20*")])

In [ ]:
# loading cslc & cmlp products
ds_cslc = xr.concat(
    [read(p, consolidated=False, as_type="xr.Dataset").sel(polarization="VV") for p in cslc_dirs], dim="acquisition_id"
).sortby("acquisition_date")

ds_cmlp = xr.concat(
    [read(p, consolidated=False, as_type="xr.Dataset").sel(polarization="VV") for p in cmlp_dirs],
    dim="acquisition_date",
).sortby("acquisition_date")
# loading PS_mask

ds_ps = read(ps_dir_name, consolidated=False)

In [ ]:
assert np.all(ds_cslc.polarization.values == ds_cmlp.polarization.values)
assert np.all(ds_cslc.acquisition_date.values == ds_cmlp.acquisition_date.values)

In [ ]:
# set velocity correction search space
abs_time = np.array(ds_cmlp.acquisition_date.values, dtype='datetime64[ns]')
time_y = xr.DataArray(((abs_time-abs_time[0]) / np.timedelta64(1, 's') / 3600 / 24 / 365.24), dims='acquisition_date', coords={'acquisition_date': abs_time})
dv = xr.DataArray(np.arange(-20., 20., 0.2), dims='vel').chunk({"vel": 1})

In [ ]:
# extract cmlp phase
cmlp_phase = ds_cmlp.phase
short_term_coh = ds_cmlp.short_term_coh
cmlp_phase = cmlp_phase.drop_vars(["slant_range_time", "azimuth_time"])

In [ ]:
## Select only valid points for all acquisition dates

In [ ]:
# identify region of valid CMLP points
mask_coh = ((short_term_coh > 0.0).all(dim="acquisition_date")).compute()
ind = np.where(mask_coh)
az_min, az_max = ind[0].min(), ind[0].max()
rg_min, rg_max = ind[1].min(), ind[1].max()

az_min_coord, az_max_coord = mask_coh.azimuth_time[az_min].values, mask_coh.azimuth_time[az_max].values
rg_min_coord, rg_max_coord = mask_coh.slant_range_time[rg_min].values, mask_coh.slant_range_time[rg_max].values
az_slice = slice(az_min_coord, az_max_coord)
rg_slice = slice(rg_min_coord, rg_max_coord)
mask_coh = mask_coh.drop_vars(mask_coh.coords)

In [ ]:
if compute_expand:
    cslc = ds_cslc.cslc.swap_dims({"acquisition_id": "acquisition_date"})
    cslc_expanded = (
        expand_cslc(cslc, posting)
        .chunk({"acquisition_date": -1, "azimuth_time": 256, "slant_range_time": 256})
    )
    cslc_expanded = write(cslc_expanded, '/home/juanma/Temp_for_ARD/Murcia/cslc_expanded')
else:
    cslc_expanded = read('/home/juanma/Temp_for_ARD/Murcia/cslc_expanded', as_type='xr.Dataset')

In [ ]:
## Definition of a zone with large deformation rate and a stable zone

In [ ]:
# print(cslc_expanded.cslc)
# print(ds_cslc.cslc)

# pri = ds_cslc.cslc[20,:,:]
# sec = ds_cslc.cslc[51,:,:]

# interferogram = get_interf(pri, sec, posting=posting)
# phase = np.angle(interferogram)

In [ ]:
# Area with subsidence
az_min_coord, az_max_coord = 920, 980
rg_min_coord, rg_max_coord = 120, 180
mask_subsidence = np.full(mask_coh.shape, False)
mask_subsidence[az_min_coord:az_max_coord, rg_min_coord:rg_max_coord] = True    

# Stable area
az_min_coord, az_max_coord = 590, 650
rg_min_coord, rg_max_coord = 800, 860
mask_stable = np.full(mask_coh.shape, False)
mask_stable[az_min_coord:az_max_coord, rg_min_coord:rg_max_coord] = True


# fig, ax = plt.subplots(figsize=(10, 10))
# im = ax.imshow(phase[az_min_coord:az_max_coord, rg_min_coord:rg_max_coord], cmap="jet", vmin=-np.pi, vmax=np.pi)
# # im = ax.imshow(phase, cmap="jet", vmin=-np.pi, vmax=np.pi)
# plt.colorbar(im, ax=ax, label="Phase (radians)")    


In [ ]:
# CMLP VS EMI

In [ ]:
if compute_emi:
    interf_emi = xr.apply_ufunc(
        emi_phase_estimation, 
        cslc_expanded.cslc,
        input_core_dims=[["acquisition_date", "win_slant_range_time", "win_azimuth_time"]],
        output_core_dims=[["acquisition_date"]],
        output_dtypes=[cslc_expanded.cslc.dtype],
        vectorize=True,
        dask="parallelized"
    )
    interf_emi = write(interf_emi, interf_emi_path)
else:
    interf_emi = read(interf_emi_path, as_type='xr.Dataset').cslc

In [ ]:
# compute phase residuals
cmlp_vs_emi = (interf_emi * xr.ufuncs.exp(-1j * cmlp_phase)).compute().chunk({"acquisition_date": 1, "azimuth_time": 256, "slant_range_time": 256})
cmlp_vs_emi["acquisition_date"] = abs_time

In [ ]:
## Temporal coherence without velocity correction

In [ ]:
temp_cohe_0 =np.abs((cmlp_vs_emi / np.abs(cmlp_vs_emi)).mean(dim="acquisition_date")).compute()

In [ ]:
## Drift estimation through temporal coherence maximization

In [ ]:
cohe_max_emi, vel_max_emi = get_cohe_vel(cmlp_vs_emi, dv, time_y, wavelength)

In [ ]:
## Phase difference statistics

## Velocity drift: CMLP vs EMI

In [ ]:
for k in range(1, 10):
    _ = xr.ufuncs.angle(cmlp_vs_emi.isel(acquisition_date=k)).where(mask_coh, np.nan).plot.hist(bins=100, alpha=0.5, density=True, histtype='step', label=f"acq_date={k}")
plt.legend()
plt.grid()

In [ ]:
## EMI vs MLC Coherence

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 8))

cohe_max_emi.where(mask_coh, np.nan).plot.hist(bins=100, alpha=0.5, label="with velocity estimation", histtype="step", range=(0.8, 1.0), ax=axs[0])
temp_cohe_0.where(mask_coh, np.nan).plot.hist(bins=100, alpha=0.5, label="whitout velocity estimation", histtype="step", range=(0.8, 1.0), ax=axs[0])

cohe_max_emi.where(mask_coh, np.nan).plot.imshow(cmap='gray', vmin=0.95, vmax=1.0, ax=axs[1])

axs[0].grid()
axs[0].legend()

In [ ]:
fig.savefig(
    os.path.join(output_plot_dir, "cmlp_vs_emi_coherence_murcia.jpg"),
    dpi=200,
    bbox_inches="tight",
)

![CMLP vs EMI](figures/cmlp_vs_emi_coherence_murcia.jpg)
*Figure 1:  Histogram (left) and image (right) of the coherence between the CMLP product and the EMI interferograms after optimization by fitting the best velocity drift between them. Coherence between both products was very high before the optimization, so it is only slightly increased.*

In [1]:
## Velocity drift

In [ ]:
import numpy as np
long_term_cohe = ds_cmlp.long_term_coh.mean(dim=('acquisition_date', 'past_reference_date'))
long_term_cohe = long_term_cohe.drop_vars(long_term_cohe.coords)

# Extract valid values
data = vel_max_emi.where(mask_coh * (long_term_cohe > 0.0)).values.ravel()
data = data[np.isfinite(data)]

mean = np.mean(data)
std = np.std(data)

fig, axs = plt.subplots(1, 2, figsize=(20, 8))

# Map
vel_max_emi.where(mask_coh * (long_term_cohe > 0.0).values, np.nan).plot.imshow(
    cmap="turbo", vmin=-20, vmax=20, ax=axs[1]
)

# Histogram
vel_max_emi.where(mask_coh * (long_term_cohe > 0.0), np.nan).plot.hist(
    bins=100,
    alpha=0.5,
    density=True,
    label="vel=opt",
    histtype="stepfilled",
    range=[-20, 20],
    ax=axs[0],
)

# Mean and std
axs[0].axvline(mean, color="red", lw=2,
               label=f"Mean = {mean:.2f}")
axs[0].axvline(mean - std, color="black", ls="--", lw=1.5,
               label=f"-1σ = {mean-std:.2f}")
axs[0].axvline(mean + std, color="black", ls="--", lw=1.5,
               label=f"+1σ = {mean+std:.2f}")

# Optional shaded ±1σ interval
axs[0].axvspan(mean - std, mean + std,
               color="gray", alpha=0.2)

axs[0].grid()
axs[0].legend()


plt.show()

In [ ]:
fig.savefig(
    os.path.join(output_plot_dir, "cmlp_vs_emi_velocity_drift_murcia.jpg"),
    dpi=200,
    bbox_inches="tight",
)

![CMLP vs EMI](figures/cmlp_vs_emi_velocity_drift_murcia.jpg)
*Figure 2:  Histogram (left) and image (right) of the velocity drift estimated by the optimization of the cohernce between the CMLP product and the EMI interferograms. The average velocity drift is 0.25 mm/year and the standard deviation is below 5 mm/year. The spatial distribution of the velocity drift does not show any specific pattern.*

In [ ]:
import numpy as np
long_term_cohe = ds_cmlp.long_term_coh.mean(dim=('acquisition_date', 'past_reference_date'))
long_term_cohe = long_term_cohe.drop_vars(long_term_cohe.coords)

# Extract valid values
data = vel_max_emi.where(mask_coh * (long_term_cohe > 0.8)).values.ravel()
data = data[np.isfinite(data)]

mean = np.mean(data)
std = np.std(data)

fig, axs = plt.subplots(1, 2, figsize=(20, 8))

# Map
vel_max_emi.where(mask_coh * (long_term_cohe > 0.8).values, np.nan).plot.imshow(
    cmap="turbo", vmin=-20, vmax=20, ax=axs[1]
)

# Histogram
vel_max_emi.where(mask_coh * (long_term_cohe > 0.8), np.nan).plot.hist(
    bins=100,
    alpha=0.5,
    density=True,
    label="vel=opt",
    histtype="stepfilled",
    range=[-20, 20],
    ax=axs[0],
)

# Mean and std
axs[0].axvline(mean, color="red", lw=2,
               label=f"Mean = {mean:.2f}")
axs[0].axvline(mean - std, color="black", ls="--", lw=1.5,
               label=f"-1σ = {mean-std:.2f}")
axs[0].axvline(mean + std, color="black", ls="--", lw=1.5,
               label=f"+1σ = {mean+std:.2f}")

# Optional shaded ±1σ interval
axs[0].axvspan(mean - std, mean + std,
               color="gray", alpha=0.2)

axs[0].grid()
axs[0].legend()

plt.show()

In [ ]:
fig.savefig(
    os.path.join(output_plot_dir, "cmlp_vs_emi_velocity_drift_masked_murcia.jpg"),
    dpi=200,
    bbox_inches="tight",
)

![CMLP vs EMI](figures/cmlp_vs_emi_velocity_drift_masked_murcia.jpg)
*Figure 3:  Histogram (left) and image (right) of the velocity drift estimated by the optimization of the cohernce between the CMLP product and the EMI interferograms but considering only pixels with average long-term coherence above 0.8. For this subset of coheremt pixels, the average velocity drift is 0.16 mm/year and the standard deviation is 1.5 mm/year. The spatial distribution of the velocity drift show only a few pixels with noticeable values departing from 0.*

## Velocity drift: CMLP vs PS

In [ ]:
cslc_mask = (
    (ds_cslc.cslc * ds_ps.mask)
    .drop_vars(["slant_range_time", "azimuth_time"])
    .swap_dims({"acquisition_id": "acquisition_date"})
)

In [ ]:
mask_ml = get_multilook(ds_ps.mask.astype(float), posting).compute()

In [ ]:
if compute_interf:
    interf = get_interf(
        cslc_mask, cslc_mask.isel(acquisition_date=0), posting
    ).compute()
    write(
        interf.chunk({"acquisition_date": 1, "azimuth_time": 4096, "slant_range_time": 4096}),
        interf_ps_path,
    )
else:
    interf = read(interf_ps_path).v

In [ ]:
## Compute difference between PS and CMLP time series

In [ ]:
mask = (interf.values != complex(0,0)) * np.isfinite(interf).values
cmlp_vs_ps = (interf * xr.ufuncs.exp(-1j * cmlp_phase)).compute().chunk({"acquisition_date": 1, "azimuth_time": 256, "slant_range_time": 256})
cmlp_vs_ps["acquisition_date"] = abs_time

In [ ]:
## Temporal coherence without velocity correction

In [ ]:
temp_cohe_0 =np.abs((cmlp_vs_ps / np.abs(cmlp_vs_ps)).mean(dim="acquisition_date")).compute()

In [ ]:
## Drift estimation through temporal coherence maximization

In [ ]:
cohe_max_ps, vel_max_ps = get_cohe_vel(cmlp_vs_ps, dv, time_y, wavelength)

In [ ]:
## Phase difference statistics

In [ ]:
for k in range(1, 10):
    _ = xr.ufuncs.angle(cmlp_vs_ps.isel(acquisition_date=k)).where(mask_ml > 0.1, np.nan).plot.hist(bins=100, alpha=0.5, density=True, histtype='step', label=f"acq_date={k}")
plt.legend()
plt.grid()

In [ ]:
## PS vs MLC Coherence

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 8))


cohe_max_ps.plot.hist(bins=100, alpha=0.5, label="with velocity estimation", histtype="step", range=(0.8, 1.0), ax=axs[0])
temp_cohe_0.plot.hist(bins=100, alpha=0.5, label="whitout velocity estimation", histtype="step", range=(0.8, 1.0), ax=axs[0])

cohe_max_ps.plot.imshow(cmap='gray', vmin=0.8, vmax=1.0, ax=axs[1])

axs[0].grid()
axs[0].legend()

In [ ]:
fig.savefig(
    os.path.join(output_plot_dir, "cmlp_vs_ps_coherence_murcia.jpg"),
    dpi=200,
    bbox_inches="tight",
)

![CMLP vs PS](figures/cmlp_vs_ps_coherence_murcia.jpg)
*Figure 4:  Histogram (left) and image (right) of the coherence between the CMLP product and the original interferograms (computed from the CSLC) at the PS locations (averaged in a window equal to the multi-look) after optimization by fitting the best velocity drift between them. Coherence between both products was very high before the optimization, but it is increased with the optimization.*

In [2]:
## Velocity drift

In [ ]:
import numpy as np

# Extract valid values
data = vel_max_ps.where(mask_coh * (mask_ml > 0.0)).values.ravel()
data = data[np.isfinite(data)]

mean = np.mean(data)
std = np.std(data)

fig, axs = plt.subplots(1, 2, figsize=(20, 8))

# Map
vel_max_ps.where(mask_coh * (mask_ml > 0.0), np.nan).plot.imshow(
    cmap="turbo", vmin=-20, vmax=20, ax=axs[1]
)

# Histogram
vel_max_ps.where(mask_coh * (mask_ml > 0.0), np.nan).plot.hist(
    bins=100,
    alpha=0.5,
    density=True,
    label="vel=opt",
    histtype="stepfilled",
    range=[-20, 20],
    ax=axs[0],
)

# Mean and std
axs[0].axvline(mean, color="red", lw=2,
               label=f"Mean = {mean:.2f}")
axs[0].axvline(mean - std, color="black", ls="--", lw=1.5,
               label=f"-1σ = {mean-std:.2f}")
axs[0].axvline(mean + std, color="black", ls="--", lw=1.5,
               label=f"+1σ = {mean+std:.2f}")

# Optional shaded ±1σ interval
axs[0].axvspan(mean - std, mean + std,
               color="gray", alpha=0.2)

axs[0].grid()
axs[0].legend()

plt.show()

In [ ]:
fig.savefig(
    os.path.join(output_plot_dir, "cmlp_vs_ps_velocity_drift_murcia.jpg"),
    dpi=200,
    bbox_inches="tight",
)

![CMLP vs PS](figures/cmlp_vs_ps_velocity_drift_murcia.jpg)
*Figure 5:  Histogram (left) and image (right) of the velocity drift estimated by the optimization of the cohernce between the CMLP product and the original interferograms at the PS locations ((averaged in a window equal to the multi-look). The average velocity drift is 0.57 mm/year and the standard deviation is below 5 mm/year. The spatial distribution of the velocity drift does not show any specific pattern.*

In [ ]:
import numpy as np

# Extract valid values
data = vel_max_ps.where(mask_coh * (mask_ml > 0.05)).values.ravel()
data = data[np.isfinite(data)]

mean = np.mean(data)
std = np.std(data)

fig, axs = plt.subplots(1, 2, figsize=(20, 8))

# Map
vel_max_ps.where(mask_coh * (mask_ml > 0.05), np.nan).plot.imshow(
    cmap="turbo", vmin=-20, vmax=20, ax=axs[1]
)

# Histogram
vel_max_ps.where(mask_coh * (mask_ml > 0.05), np.nan).plot.hist(
    bins=100,
    alpha=0.5,
    density=True,
    label="vel=opt",
    histtype="stepfilled",
    range=[-20, 20],
    ax=axs[0],
)

# Mean and std
axs[0].axvline(mean, color="red", lw=2,
               label=f"Mean = {mean:.2f}")
axs[0].axvline(mean - std, color="black", ls="--", lw=1.5,
               label=f"-1σ = {mean-std:.2f}")
axs[0].axvline(mean + std, color="black", ls="--", lw=1.5,
               label=f"+1σ = {mean+std:.2f}")

# Optional shaded ±1σ interval
axs[0].axvspan(mean - std, mean + std,
               color="gray", alpha=0.2)

axs[0].grid()
axs[0].legend()

plt.show()

In [ ]:
fig.savefig(
    os.path.join(output_plot_dir, "cmlp_vs_ps_velocity_drift_masked_murcia.jpg"),
    dpi=200,
    bbox_inches="tight",
)

![CMLP vs PS](figures/cmlp_vs_ps_velocity_drift_masked_murcia.jpg)
*Figure 6:  Histogram (left) and image (right) of the velocity drift estimated by the optimization of the cohernce between the CMLP product and the original interferograms at the PS positions (averaged in a window equal to the multi-look) but considering only cases with at least 5\% of PS per window. In this case, the average velocity drift is 0.16 mm/year and the standard deviation is below 1 mm/year. The spatial distribution of the velocity drift does not show pixels with noticeable values departing from 0.*

## Lag-1 Accumulated Drift Analysis

In [ ]:
# From Francesco: trying to catch the bias by other means
# Averaging and accumulating the lag-1 differences between EMI and CMLP

n_acquisitions = len(time_y)

d_phi = np.zeros((n_acquisitions), dtype=np.float64)

interf_mask2 = np.where(mask_coh * (long_term_cohe > 0.2), 1.0, np.nan)

for n in range(n_acquisitions-1):
    print(f"\r{n + 1} of {n_acquisitions-1}", end="", flush=True)
    cmlp_ph = cmlp_phase[n,:,:].where(mask_coh * (long_term_cohe > 0.0), np.nan).values
    cmlp_ph_ = cmlp_phase[n+1,:,:].where(mask_coh * (long_term_cohe > 0.0), np.nan).values

    ph = np.squeeze(np.asarray(cmlp_ph)) - np.squeeze(np.asarray(cmlp_ph_)) - np.angle(interf_emi[:, :, n]) +  np.angle(interf_emi[:, :, n+1])   # (n_az, n_rg)
    d_phi[n] = np.angle( np.nanmean(np.exp(1j * ph) * interf_mask2) )

In [ ]:
# Plotting the cumulative sum of the lag-1 phase differences, which should correspond to the average time series

fig, axs = plt.subplots(figsize=(10, 6))
axs.plot(time_y, np.cumsum(d_phi) * wavelength / (4*np.pi))
axs.set_xlabel("Time (years)")
axs.set_ylabel("(mm)")
axs.grid()


In [ ]:
fig.savefig(
    os.path.join(output_plot_dir, "cmlp_vs_emi_lag1_accumulated_drift_murcia.jpg"),
    dpi=200,
    bbox_inches="tight",
)

![CMLP vs EMI](figures/cmlp_vs_emi_lag1_accumulated_drift_murcia.jpg)
*Figure 7:  Temporal evolution of the accumulated drift of the lag-1 difference in the interferograms provided by CMLP and EMI, computed over the whole scene. All values are below 3 mm. No trend is noticed.*

In [ ]:
# Area with strong subsidence
# Averaging and accumulating the lag-1 differences between EMI and CMLP

n_acquisitions = len(time_y)

d_phi_subsidence = np.zeros((n_acquisitions), dtype=np.float64)
d_phi_stable = np.zeros((n_acquisitions), dtype=np.float64)

interf_mask_subsidence = np.where(mask_coh * (long_term_cohe > 0.2) * mask_subsidence, 1.0, np.nan)
interf_mask_stable = np.where(mask_coh * (long_term_cohe > 0.2) * mask_stable, 1.0, np.nan)

for n in range(n_acquisitions-1):
    print(f"\r{n + 1} of {n_acquisitions-1}", end="", flush=True)
    cmlp_ph = cmlp_phase[n,:,:].where(mask_coh * (long_term_cohe > 0.0), np.nan).values
    cmlp_ph_ = cmlp_phase[n+1,:,:].where(mask_coh * (long_term_cohe > 0.0), np.nan).values

    ph = np.squeeze(np.asarray(cmlp_ph)) - np.squeeze(np.asarray(cmlp_ph_)) - np.angle(interf_emi[:, :, n]) +  np.angle(interf_emi[:, :, n+1])   # (n_az, n_rg)
    
    d_phi_subsidence[n] = np.angle( np.nanmean(np.exp(1j * ph) * interf_mask_subsidence) )
    d_phi_stable[n] = np.angle( np.nanmean(np.exp(1j * ph) * interf_mask_stable) )

In [ ]:
# Plotting the cumulative sum of the lag-1 phase differences, which should correspond to the average time series

fig, axs = plt.subplots(figsize=(10, 6))
axs.plot(time_y, np.cumsum(d_phi_subsidence) * wavelength / (4*np.pi))
axs.set_xlabel("Time (years)")
axs.set_ylabel("(mm)")
axs.grid()

In [ ]:
fig.savefig(
    os.path.join(output_plot_dir, "cmlp_vs_emi_lag1_accumulated_drift_at_subsidence_area_murcia.jpg"),
    dpi=200,
    bbox_inches="tight",
)

![CMLP vs EMI](figures/cmlp_vs_emi_lag1_accumulated_drift_at_subsidence_area_murcia.jpg)
*Figure 8:  Temporal evolution of the accumulated drift of the lag-1 difference in the interferograms provided by CMLP and EMI, computed over a zone with large subsidence. All values are below 2 mm. No trend is noticed.*

In [ ]:
# Plotting the cumulative sum of the lag-1 phase differences, which should correspond to the average time series

fig, axs = plt.subplots(figsize=(10, 6))
axs.plot(time_y, np.cumsum(d_phi_stable) * wavelength / (4*np.pi))
axs.set_xlabel("Time (years)")
axs.set_ylabel("(mm)")
axs.grid()

In [ ]:
fig.savefig(
    os.path.join(output_plot_dir, "cmlp_vs_emi_lag1_accumulated_drift_at_stable_area_murcia.jpg"),
    dpi=200,
    bbox_inches="tight",
)

![CMLP vs EMI](figures/cmlp_vs_emi_lag1_accumulated_drift_at_stable_area_murcia.jpg)
*Figure 9:  Temporal evolution of the accumulated drift of the lag-1 difference in the interferograms provided by CMLP and EMI, computed over a zone with large subsidence. All values are below 2 mm. No trend is noticed.*

## Main Results
In all cases we have found agreement, with shifts **below the prescribed quality threshold**, for all comparisons of CMLP against EMI and PS.

The accumulated lag-1 interferograms **do not show any temporal pattern** in any cases (evaluated as a whole or in specific zones).

## Final Conclusion
The GMLP products derived from SAOCOM in this area **comply with the expected quality criteria**.